In [2]:
from langchain_community.document_loaders import PyPDFLoader

# grap ka pdf load kiya
loader = PyPDFLoader("../docs/grap.pdf")
pages = loader.load()
print(len(pages), "pages")
print(pages[0].page_content[:500])

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 78 0 (offset 0)
Ignoring wrong pointing object 94 0 (offset 0)
Ignoring wrong pointing object 144 0 (offset 0)


17 pages
COMMISSION FOR AIR QUALITY MANAGEMENT 
IN NATIONAL CAPITAL REGION AND ADJOINING AREAS  
GRADED RESPONSE ACTION PLAN (GRAP) 
FOR THE NATIONAL CAPITAL REGION (NCR)  
(Revision: 29.09.2026) 
 
1. The GRAP for the entire NCR has been classified under 4 different stages of 
adverse air quality in Delhi viz. Stage – I ‘Poor’ (AQI 201 – 300), Stage – II ‘Very Poor 
(AQI 301-400), Stage – III ‘Severe’ (AQI 401-450) and Stage – IV ‘Severe +’ (AQI >450) 
respectively.  
2. Based on the dynamic model and w


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chhote chunks, thoda overlap ke saath
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(pages)
print(len(chunks), "chunks")
print(chunks[5].page_content)

46 chunks
projects with plot size equal to or more than 
500 sqm , which are not registered on the 
‘web portal’ of the respective State / GNCTD 
and / or which do not fulfil the other 
requirements as per the above noted 
statutory Directions, for remote monitoring of 
dust mitigation measures.  
- Chairpersons/ Member Secretaries  – CPCB, 
DPCC, SPCBs (NCR). 
- Commissioners / Chief Engineers of Urban 
Local Bodies in Delhi-NCR towns. 
- Construction agencies and plot owners (both 
public & private). 
3. Ensure regular lifting of Municipal Solid 
Waste (MSW), C&D waste, Industrial and 
Hazardous waste from dedicated dump 
sites/ industrial areas  and ensure that no 
waste is dumped illegally in open land areas. 
- Chairpersons/ Member Secretaries  – CPCB, 
DPCC, SPCBs (NCR).


In [4]:
# vscode ke link chunk ke andar to nahi hain, check
print(any("vscode" in c.page_content for c in chunks))

False


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# chhota free embedding model, laptop pe chal jata hai
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# chunks ko vector bana ke chroma mein store
db = Chroma.from_documents(chunks, emb, persist_directory="chroma_grap")
print(db._collection.count())

/Users/najeebahmad/Desktop/AQI-PRED-RAG/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11823.02it/s]


46


In [6]:
queries = ["what actions are taken under stage III severe air quality",
           "diesel generator ban in NCR"]

for q in queries:
    print("QUESTION:", q)
    for d in db.similarity_search(q, k=3):
        print("page", d.metadata.get("page"))
        print(d.page_content[:300])
        print("-----")

QUESTION: what actions are taken under stage III severe air quality
page 0
stage, also provided that the higher projected AQI levels are likely to sustain for 
longer period.    
3. Even if the AQI forecast does not indicate the AQI of Delhi to be breaching a 
particular threshold and under extreme meteorological conditions or due to any 
episodic event the AQI breaches th
-----
page 0
likely to fall in the ‘Severe’ or ‘Severe +’ category (Stage III and beyond).  
7. The Commission may decide upon any exceptions and additional measures to 
the schedule of the GRAP, under different air pollution categories i.e., Stages I to 
IV, as per the prevalent AQI and weather forecast.
-----
page 0
COMMISSION FOR AIR QUALITY MANAGEMENT 
IN NATIONAL CAPITAL REGION AND ADJOINING AREAS  
GRADED RESPONSE ACTION PLAN (GRAP) 
FOR THE NATIONAL CAPITAL REGION (NCR)  
(Revision: 29.09.2026) 
 
1. The GRAP for the entire NCR has been classified under 4 different stages of 
adverse air quality in Delhi v
---

In [7]:
import re

# har page pe jahan "stage" likha hai wo chhoti lines (header pakadne ke liye)
for i, p in enumerate(pages):
    lines = [l.strip() for l in p.page_content.split("\n")]
    hits = [l for l in lines if re.search(r"stage", l, re.I) and len(l) < 90]
    print("page", i, "->", hits[:6])

page 0 -> ['1. The GRAP for the entire NCR has been classified under 4 different stages of', 'adverse air quality in Delhi viz. Stage – I ‘Poor’ (AQI 201 – 300), Stage – II ‘Very Poor', '(AQI 301-400), Stage – III ‘Severe’ (AQI 401-450) and Stage – IV ‘Severe +’ (AQI >450)', 'on a day-to-day basis, actions under Stages I, II, III and IV of the GRAP shall be', 'stage, also provided that the higher projected AQI levels are likely to sustain for', 'episodic event the AQI breaches the threshold, that particular stage of the GRAP']
page 1 -> ['Stage I – ‘Poor’ Air Quality']
page 2 -> []
page 3 -> []
page 4 -> []
page 5 -> []
page 6 -> []
page 7 -> ['• Check AQI from SAMEER App and stage of GRAP in force from CAQM website.']
page 8 -> ['Stage II – ‘Very Poor’ Air Quality']
page 9 -> []
page 10 -> ['Stage III – ‘Severe’ Air Quality']
page 11 -> []
page 12 -> []
page 13 -> []
page 14 -> []
page 15 -> ['Stage IV – ‘Severe +’ Air Quality', '3. Ban C&D activities, as in the GRAP Stage- III,']
pag

In [8]:
import re
from collections import Counter
from langchain_core.documents import Document

# header jaise "Stage III – 'Severe' Air Quality" (line ke shuru mein)
hdr = re.compile(r"^\s*Stage\s+(IV|III|II|I)\s*[–-]\s*[‘']", re.M)

tagged = []
cur = "General"
for i, p in enumerate(pages):
    txt = p.page_content
    pos = 0
    for m in hdr.finditer(txt):
        # header se pehle ka text purane stage ka hai
        if m.start() > pos:
            tagged.append(Document(page_content=txt[pos:m.start()], metadata={"page": i, "stage": cur}))
        cur = "Stage " + m.group(1)
        pos = m.start()
    tagged.append(Document(page_content=txt[pos:], metadata={"page": i, "stage": cur}))

print(Counter((d.metadata["page"], d.metadata["stage"]) for d in tagged))

Counter({(0, 'General'): 1, (1, 'General'): 1, (1, 'Stage I'): 1, (2, 'Stage I'): 1, (3, 'Stage I'): 1, (4, 'Stage I'): 1, (5, 'Stage I'): 1, (6, 'Stage I'): 1, (7, 'Stage I'): 1, (8, 'Stage II'): 1, (9, 'Stage II'): 1, (10, 'Stage III'): 1, (11, 'Stage III'): 1, (12, 'Stage III'): 1, (13, 'Stage III'): 1, (14, 'Stage III'): 1, (15, 'Stage IV'): 1, (16, 'Stage IV'): 1})


In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks2 = splitter.split_documents(tagged)

# har chunk ke aage stage ka naam jod diya
for c in chunks2:
    c.page_content = f"[GRAP {c.metadata['stage']}] " + c.page_content

print(len(chunks2))
print(Counter(c.metadata["stage"] for c in chunks2))

db2 = Chroma.from_documents(chunks2, emb, persist_directory="chroma_grap_v2")
print(db2._collection.count())

47
Counter({'Stage I': 19, 'Stage III': 12, 'Stage II': 6, 'General': 5, 'Stage IV': 5})
47


In [10]:
def show(res):
    for d in res:
        print(d.metadata["stage"], "| page", d.metadata["page"], "|", d.page_content[:200].replace("\n", " "))
        print("-----")

q = "what actions are taken under stage III severe air quality"
print("STAGE III, bina filter")
show(db2.similarity_search(q, k=3))

print("STAGE III, filter ke saath")
show(db2.similarity_search(q, k=3, filter={"stage": "Stage III"}))

q2 = "diesel generator ban in NCR"
print("DIESEL GENERATOR, bina filter")
show(db2.similarity_search(q2, k=3))

STAGE III, bina filter
General | page 0 | [GRAP General] stage, also provided that the higher projected AQI levels are likely to sustain for  longer period.     3. Even if the AQI forecast does not indicate the AQI of Delhi to be breaching a 
-----
General | page 0 | [GRAP General] COMMISSION FOR AIR QUALITY MANAGEMENT  IN NATIONAL CAPITAL REGION AND ADJOINING AREAS   GRADED RESPONSE ACTION PLAN (GRAP)  FOR THE NATIONAL CAPITAL REGION (NCR)   (Revision: 29.09.2026
-----
Stage III | page 10 | [GRAP Stage III] Stage III – ‘Severe’ Air Quality   (DELHI AQI ranging between 401-450)   Actions  Agencies responsible / Implementing  Agencies  1. Construction & Demolition activities:    (i)  Enfor
-----
STAGE III, filter ke saath
Stage III | page 10 | [GRAP Stage III] Stage III – ‘Severe’ Air Quality   (DELHI AQI ranging between 401-450)   Actions  Agencies responsible / Implementing  Agencies  1. Construction & Demolition activities:    (i)  Enfor
-----
Stage III | page 11 | [GRAP Stage III] t

In [11]:
# kisi bhi page mein aisi lines jo stages ke build-up ki baat karti hon
keys = ["in addition", "earlier stage", "lower stage", "previous stage", "continue", "stage-i,", "stage- iii", "all actions"]
for i, p in enumerate(pages):
    for line in p.page_content.split("\n"):
        if any(k in line.lower() for k in keys):
            print("page", i, "|", line.strip())

page 0 | 4. Restrictive actions undertaken as per previous stages shall be continued, in
page 0 | invoked, shall be in addition to those under Stage I and II respectively and so on
page 11 | be permitted to be continued in the NCR, subject
page 11 | under 1(i) above , shall be continued to be
page 15 | 3. Ban C&D activities, as in the GRAP Stage- III,


In [12]:
# pdf ke point 1 ke thresholds
def grap_stage(aqi):
    if aqi > 450: return "Stage IV"
    if aqi > 400: return "Stage III"
    if aqi > 300: return "Stage II"
    if aqi > 200: return "Stage I"
    return None

for a in [150, 250, 350, 420, 480]:
    print(a, grap_stage(a))

# us stage ke chunks plus general rules
def get_context(stage, question, k=6):
    gen = db2.similarity_search(question, k=2, filter={"stage": "General"})
    docs = db2.similarity_search(question, k=k, filter={"stage": stage})
    return gen + docs

ctx = get_context("Stage II", "diesel generator ban")
for d in ctx:
    print(d.metadata["stage"], "| page", d.metadata["page"], "|", d.page_content[:250].replace("\n", " "))
    print("-----")

150 None
250 Stage I
350 Stage II
420 Stage III
480 Stage IV
General | page 0 | [GRAP General] stage, also provided that the higher projected AQI levels are likely to sustain for  longer period.     3. Even if the AQI forecast does not indicate the AQI of Delhi to be breaching a  particular threshold and under extreme meteorolog
-----
General | page 0 | [GRAP General] likely to fall in the ‘Severe’ or ‘Severe +’ category (Stage III and beyond).   7. The Commission may decide upon any exceptions and additional measures to  the schedule of the GRAP, under different air pollution categories i.e., Stage
-----
Stage II | page 9 | [GRAP Stage II] • Regularly replace air filters at recommended intervals in your vehicles.   • Avoid dust generating construction activities during the months of October to  January.  • Ensure cleaning of kitchen chimney filters at regular intervals.
-----
Stage II | page 8 | [GRAP Stage II] Stage II – ‘Very Poor’ Air Quality   (DELHI AQI ranging between 301-400)  

In [13]:
stage_order = ["Stage I", "Stage II", "Stage III", "Stage IV"]

def get_context(stage, question, k=8):
    # pdf point 4: upar ke stage mein neeche ke stages ki actions bhi lagu hoti hain
    upto = stage_order[: stage_order.index(stage) + 1]
    gen = db2.similarity_search(question, k=2, filter={"stage": "General"})
    docs = db2.similarity_search(question, k=k, filter={"stage": {"$in": upto}})
    return gen + docs

ctx = get_context("Stage II", "diesel generator ban")
for d in ctx:
    print(d.metadata["stage"], "| page", d.metadata["page"], "|", d.page_content[:250].replace("\n", " "))
    print("-----")

General | page 0 | [GRAP General] stage, also provided that the higher projected AQI levels are likely to sustain for  longer period.     3. Even if the AQI forecast does not indicate the AQI of Delhi to be breaching a  particular threshold and under extreme meteorolog
-----
General | page 0 | [GRAP General] likely to fall in the ‘Severe’ or ‘Severe +’ category (Stage III and beyond).   7. The Commission may decide upon any exceptions and additional measures to  the schedule of the GRAP, under different air pollution categories i.e., Stage
-----
Stage I | page 3 | [GRAP Stage I] Explosive Safety Organizations (PESO).  16. DISCOMs to minimise power supply  interruptions in NCR.   - Head of Power distribution companies in  Delhi-NCR.  17. Ensure that diesel generator sets are not  used as regular source of power
-----
Stage I | page 4 | [GRAP Stage I] etc. in accordance with Direction No. 76  dated 29.09.2023 with amendments, if any.  Capacity Range  of DG sets  System to be  adopted for

In [15]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq

# .env file project folder mein kahin bhi ho, wahan se dhundh lega
load_dotenv(find_dotenv())
print("key found:", bool(os.getenv("GROQ_API_KEY")))

key found: True


In [18]:
# model ka naam groq console ke models page se copy karke yahan daalo
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0)

def answer(aqi_24h, question):
    stage = grap_stage(aqi_24h)
    if stage is None:
        return "predicted aqi 200 se neeche hai, grap ka koi stage lagu nahi hoga."

    ctx = get_context(stage, question)
    context_text = "\n\n".join(d.page_content for d in ctx)

    prompt = f"""you explain the GRAP document for delhi-ncr in simple english.
predicted 24-hour average AQI is {aqi_24h:.0f}, which means {stage} (stage already decided from the document thresholds).
answer only from the context below. if the answer is not in the context, say it was not found in the document.
the document says actions of lower stages continue along with the current stage, mention this when it matters.

context:
{context_text}

question: {question}"""
    return llm.invoke(prompt).content

In [19]:
print(answer(350, "can i use diesel generators?"))
print("=========")
print(answer(420, "which construction activities are banned?"))
print("=========")
print(answer(150, "what restrictions apply?"))

**Answer**

No – under the current Stage II (AQI ≈ 350) you are not allowed to use diesel generators for normal use.  
The rules from Stage I still apply, and Stage I says:

* Portable diesel generators below 19 kW (23 kVA) are **not generally permitted** – they may only be used for the prescribed emergency services (e.g., hospitals, railways, airports, etc.).  
* Larger diesel generators (800 kW/1000 kVA and above) can be used only if they meet the emission‑control standards set in Direction No. 76; otherwise they are also restricted.

So unless your generator is one of the emergency‑service types or meets the required emission‑control standards, you cannot use it.
**Construction activities that are banned (Stage III – “Severe” air quality)**  

- Earthwork for excavation and filling, including boring and drilling.  
- Piling works.  
- All demolition works.  
- Laying of sewer lines, water lines, drainage and electric cabling by open‑trench methods.  
- Brick or masonry works.  
- Op

In [20]:
def answer(aqi_24h, question, show_sources=True):
    stage = grap_stage(aqi_24h)
    if stage is None:
        return "predicted aqi 200 se neeche hai, grap ka koi stage lagu nahi hoga."

    ctx = get_context(stage, question, k=10)
    # har chunk ke saath stage aur pdf page number
    context_text = "\n\n".join(
        f"[{d.metadata['stage']}, pdf page {d.metadata['page'] + 1}]\n{d.page_content}" for d in ctx
    )

    prompt = f"""you explain the GRAP document for delhi-ncr in simple english.
predicted 24-hour average AQI is {aqi_24h:.0f}, which means {stage} (already decided from the document thresholds).

rules:
- answer only from the context below. if something is not in the context, say it was not found.
- actions of lower stages continue along with the current stage, mention this when it matters.
- if the context lists exemptions or exceptions (projects or activities allowed to continue), always mention them together with the ban.
- the context comes from a pdf, so tables can look garbled. do not guess numbers or capacity limits from garbled table text. if unsure, say to check the table on that pdf page.
- keep the answer short, use bullet points.

context:
{context_text}

question: {question}"""

    out = llm.invoke(prompt).content
    if show_sources:
        used = sorted({(d.metadata["stage"], d.metadata["page"] + 1) for d in ctx})
        out += "\n\nsources used: " + ", ".join(f"{s} p{p}" for s, p in used)
    return out

In [21]:
print(answer(350, "can i use diesel generators?"))
print("=========")
print(answer(420, "which construction activities are banned and which are allowed?"))
print("=========")
print(answer(480, "are trucks allowed to enter delhi?"))

- **Stage II (AQI 350)** does not add any new rules about diesel generators.  
- **Stage I rules still apply** because lower‑stage actions continue.  
- **Regular use of diesel generators is prohibited** in all sectors (industrial, commercial, residential).  
- **Exemptions**: diesel generators may be used only for the prescribed emergency services listed in Stage I (e.g., hospitals, railways, metro, airports, sewage treatment, national‑security projects, telecommunications, etc.).  
- No other activities are allowed to use diesel generators under Stage II.

sources used: General p1, Stage I p3, Stage I p4, Stage I p5, Stage I p6, Stage I p8, Stage II p10
**Construction activities under Stage III (AQI 420 – “Severe”)**

- **Banned / strictly restricted**  
  - Earthwork for excavation & filling (including boring & drilling)  
  - Piling works  
  - All demolition works  
  - Open‑trench laying of sewer, water, drainage or electric cabling  
  - Brick / masonry works  
  - Operation of 